# Weighted Lebesgue Spaces and Bessel-Sobolev Priors

This notebook demonstrates the new `WeightedLebesgue` space and how inner-product weighting affects
Laplacian-based priors via `BesselSobolevInverse` covariances.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from intervalinf import IntervalDomain, Function
from intervalinf.spaces import Lebesgue, WeightedLebesgue
from intervalinf.core.boundary import BoundaryConditions
from intervalinf.core.config import IntegrationConfig
from intervalinf.operators import Laplacian, BesselSobolevInverse

print("Imports successful!")

Imports successful!


## 1. Create Plain and Weighted Lebesgue Spaces

In [10]:
domain = IntervalDomain(0.1, 1.0)
bc = BoundaryConditions.dirichlet()
integration_config = IntegrationConfig(method="simpson", n_points=2000)

# Plain L² space
space_plain = Lebesgue(0, domain, basis=None, integration_config=integration_config)

# Weighted L²(r²) space
def weight_r2(r):
    return np.asarray(r, dtype=float) ** 2

space_weighted = WeightedLebesgue(0, domain, weight_r2, integration_config=integration_config)

print(f"Plain L² space created")
print(f"Weighted L²(r²) space created")

Plain L² space created
Weighted L²(r²) space created


## 2. Compare Inner Products

In [11]:
f = Function(domain, evaluate_callable=lambda r: np.sin(np.pi * np.asarray(r)))
g = Function(domain, evaluate_callable=lambda r: np.cos(2 * np.pi * np.asarray(r)))

inner_plain = space_plain.inner_product(f, g)
inner_weighted = space_weighted.inner_product(f, g)

print(f"⟨f, g⟩ (plain L²): {inner_plain:.6f}")
print(f"⟨f, g⟩_w (weighted): {inner_weighted:.6f}")
print(f"Difference: {abs(inner_plain - inner_weighted):.6f}")

⟨f, g⟩ (plain L²): -0.226286
⟨f, g⟩_w (weighted): -0.044057
Difference: 0.182229


## 3. Create Laplacians

In [12]:
# Laplacian on plain L²
L_plain = Laplacian(space_plain, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

# Laplacian on weighted L²(r²)
L_weighted = Laplacian(space_weighted, bc, 1.0, method="spectral", dofs=20, integration_config=integration_config)

print("Laplacians created.")
print(f"\nFirst 3 eigenvalues:")
for i in range(3):
    print(f"  λ_{i}: {L_plain.get_eigenvalue(i):.4f}")

Laplacians created.

First 3 eigenvalues:
  λ_0: 12.1847
  λ_1: 48.7388
  λ_2: 109.6623


## 4. Create Bessel-Sobolev Covariances

In [13]:
k, s = 1.5, 1.0

C_plain = BesselSobolevInverse(domain=space_plain, codomain=space_plain, k=k, s=s, 
    L=L_plain, dofs=20, integration_config=integration_config)

C_weighted = BesselSobolevInverse(domain=space_weighted, codomain=space_weighted, k=k, s=s, 
    L=L_weighted, dofs=20, integration_config=integration_config)

print(f"Bessel-Sobolev C = (k² I - Δ)^{{-s}} created")
print(f"  k={k}, s={s}")
print(f"  Plain: uses fast transforms = {C_plain._can_use_fast_transforms}")
print(f"  Weighted: uses radial fast path = {C_weighted._radial_dirichlet_fast}")

Bessel-Sobolev C = (k² I - Δ)^{-s} created
  k=1.5, s=1.0
  Plain: uses fast transforms = True
  Weighted: uses radial fast path = False


## 5. Compare Prior Standard Deviations

In [ ]:
r_fine = np.linspace(0.12, 0.98, 200)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Eigenvalues comparison
eigenvalues_plain = [L_plain.get_eigenvalue(j) for j in range(10)]
eigenvalues_weighted = [L_weighted.get_eigenvalue(j) for j in range(10)]

axes[0, 0].semilogy(range(10), eigenvalues_plain, 'bo-', linewidth=2, markersize=8, label='Plain L²')
axes[0, 0].semilogy(range(10), eigenvalues_weighted, 'rs--', linewidth=2, markersize=8, label='Weighted L²(r²)')
axes[0, 0].set_xlabel('Mode index j')
axes[0, 0].set_ylabel('Eigenvalue λ_j')
axes[0, 0].set_title('Laplacian Eigenvalues')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# First three eigenfunctions (plain)
axes[0, 1].set_title('Eigenfunctions on Plain L²')
for j in range(3):
    phi_j = L_plain.get_eigenfunction(j)
    phi_vals = np.array([phi_j(r) for r in r_fine])
    axes[0, 1].plot(r_fine, phi_vals, linewidth=2, label=f'φ₀ (λ={eigenvalues_plain[j]:.1f})')
axes[0, 1].set_xlabel('r')
axes[0, 1].set_ylabel('φ_j(r)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# First three eigenfunctions (weighted)
axes[1, 0].set_title('Eigenfunctions on Weighted L²(r²)')
for j in range(3):
    phi_j = L_weighted.get_eigenfunction(j)
    phi_vals = np.array([phi_j(r) for r in r_fine])
    axes[1, 0].plot(r_fine, phi_vals, linewidth=2, label=f'φ_{j} (λ={eigenvalues_weighted[j]:.1f})')
axes[1, 0].set_xlabel('r')
axes[1, 0].set_ylabel('φ_j(r)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Orthonormality check: norms w.r.t. each space's inner product
plain_norms = [space_plain.inner_product(L_plain.get_eigenfunction(j), L_plain.get_eigenfunction(j)) for j in range(6)]
weighted_norms = [space_weighted.inner_product(L_weighted.get_eigenfunction(j), L_weighted.get_eigenfunction(j)) for j in range(6)]

x = np.arange(6)
width = 0.35
axes[1, 1].bar(x - width/2, plain_norms, width, label='Plain L²', color='b', alpha=0.7)
axes[1, 1].bar(x + width/2, weighted_norms, width, label='Weighted L²(r²)', color='r', alpha=0.7)
axes[1, 1].set_xlabel('Mode index j')
axes[1, 1].set_ylabel('‖φ_j‖² (w.r.t. space inner product)')
axes[1, 1].set_title('Eigenfunction Normalization')
axes[1, 1].set_xticks(x)
axes[1, 1].legend()
axes[1, 1].axhline(y=1.0, color='k', linestyle='--', alpha=0.3)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Eigenvalues (first 5 modes):")
print(f"  j=0: plain={eigenvalues_plain[0]:.4f}, weighted={eigenvalues_weighted[0]:.4f}")
print(f"  j=1: plain={eigenvalues_plain[1]:.4f}, weighted={eigenvalues_weighted[1]:.4f}")
print(f"  j=2: plain={eigenvalues_plain[2]:.4f}, weighted={eigenvalues_weighted[2]:.4f}")
print(f"\n✅ Note: eigenvalues are identical — the Laplacian spectrum is unchanged by weighting")

## Summary

✅ **WeightedLebesgue** implements $L^2(w)$ as a unified mechanism via `MassWeightedHilbertSpace`

✅ **Weighted priors** automatically encode geometric structure (e.g., spherical shells with $w(r)=r^2$)

✅ **Transparent to operators** — the same `BesselSobolevInverse` code handles both plain and weighted domains

✅ **Self-adjoint** on the space's inner product, whether plain or weighted